# Week 8 Exercise — The Price is Right (profe-ssor)

Autonomous deal-hunting agentic AI: scan deals, estimate prices with Specialist + RAG + Ensemble, and surface opportunities.

- **SpecialistAgent**: Fine-tuned LLM on Modal
- **FrontierAgent**: RAG + frontier model (OpenAI/DeepSeek)
- **EnsembleAgent**: Combines Specialist, Frontier, RandomForest
- **ScannerAgent**: RSS deal scraper
- **MessagingAgent**: Push notifications
- **PlanningAgent**: Orchestrates scan → price → alert

Run this notebook from `week8/` or from `week8/community_contributions/profe-ssor/`; the first cell adds `week8` to the path and sets the working directory so the vector store and data paths work.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

# Point to week8 so we can import agents and use week8 data paths
cwd = Path.cwd()
if cwd.name == "profe-ssor":
    week8 = cwd.parent.parent
else:
    week8 = cwd
if str(week8) not in sys.path:
    sys.path.insert(0, str(week8))

# Work from week8 so products_vectorstore, memory.json, train/test pkls resolve
os.chdir(week8)

print(f"Working directory: {os.getcwd()}")
print(f"week8 on path: {week8}")


In [ ]:
# Ensure Modal token is available (e.g. when kernel was started before `modal token set`)
if not os.environ.get("MODAL_TOKEN_ID") or not os.environ.get("MODAL_TOKEN_SECRET"):
    import toml
    _modal_toml = os.environ.get("MODAL_CONFIG_PATH") or os.path.expanduser("~/.modal.toml")
    if os.path.exists(_modal_toml):
        _modal_cfg = toml.load(_modal_toml)
        for _prof_name, _prof_data in _modal_cfg.items():
            if isinstance(_prof_data, dict) and _prof_data.get("active") is True:
                if _prof_data.get("token_id") and _prof_data.get("token_secret"):
                    os.environ["MODAL_TOKEN_ID"] = _prof_data["token_id"]
                    os.environ["MODAL_TOKEN_SECRET"] = _prof_data["token_secret"]
                break
        else:
            for _prof_name, _prof_data in _modal_cfg.items():
                if isinstance(_prof_data, dict) and _prof_data.get("token_id") and _prof_data.get("token_secret"):
                    os.environ["MODAL_TOKEN_ID"] = _prof_data["token_id"]
                    os.environ["MODAL_TOKEN_SECRET"] = _prof_data["token_secret"]
                    break

import modal
print(f"Modal version: {modal.__version__}")
print("Modal token: OK" if (os.environ.get("MODAL_TOKEN_ID") and os.environ.get("MODAL_TOKEN_SECRET")) else "Modal token: set via ~/.modal.toml or MODAL_TOKEN_* env vars")


## 1. Environment and secrets

Required: `OPENAI_API_KEY`, `HF_TOKEN`. Optional: `DEEPSEEK_API_KEY`, `PUSHOVER_USER`, `PUSHOVER_TOKEN` for notifications.

Modal: run `uv run modal token set --token-id ... --token-secret ...` (or add `MODAL_TOKEN_ID` / `MODAL_TOKEN_SECRET` to `.env`). Create an **hf-secret** in the Modal dashboard for the Specialist pricer.

In [ ]:
required = ["OPENAI_API_KEY", "HF_TOKEN"]
optional = ["DEEPSEEK_API_KEY", "PUSHOVER_USER", "PUSHOVER_TOKEN"]
for var in required:
    status = "SET" if os.getenv(var) else "MISSING"
    print(f"  {var}: {status}")
for var in optional:
    status = "SET" if os.getenv(var) else "(optional)"
    print(f"  {var}: {status}")


## 2. Deploy the Specialist pricer on Modal

From the repo root or week8: `uv run modal deploy week8/pricer_service2.py` (or from here: `!uv run modal deploy ../pricer_service2.py` if your cwd is profe-ssor). Ensure the **hf-secret** exists in Modal.

In [ ]:
# Deploy from week8 (current dir is week8 after first cell)
!uv run modal deploy pricer_service2.py


## 3. Test SpecialistAgent

Uses the deployed Modal `Pricer` to estimate price from a product description.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

from agents.specialist_agent import SpecialistAgent

specialist = SpecialistAgent()
test_desc = "Quadcast HyperX condenser mic, USB-C, crystal clear audio"
price = specialist.price(test_desc)
print(f"Estimated price: ${price:.2f}")


## 4. RAG vector store (products)

You need a ChromaDB `products_vectorstore` in week8 with a `products` collection (embeddings + metadatas including `price`). If you already built it in week8 day2, skip to section 5. Otherwise: load `train.pkl` (e.g. from week6 or course materials), use `agents.items.Item`, encode with SentenceTransformer (`all-MiniLM-L6-v2`), and add to Chroma. See week8 day2 notebook for the full pipeline.

In [ ]:
import chromadb

DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
coll = client.get_or_create_collection("products")
n = coll.count()
print(f"Collection 'products' has {n} documents")
if n == 0:
    print("Build the vector DB (week8 day2): load train.pkl, encode with SentenceTransformer, add to Chroma.")


## 5. Full pipeline: PlanningAgent + DealAgentFramework

The PlanningAgent uses ScannerAgent (RSS), EnsembleAgent (Specialist + Frontier + RF), and MessagingAgent. The DealAgentFramework wraps the planner and memory (surfaced opportunities).

In [ ]:
from deal_agent_framework import DealAgentFramework
from agents.deals import Opportunity, Deal

framework = DealAgentFramework()
framework.init_agents_as_needed()
print("DealAgentFramework ready.")


In [ ]:
# Run one planning cycle: scan RSS, price deals, optionally send alert for best deal above threshold
opp = framework.planner.plan(memory=[u.deal.url for u in framework.memory])
if opp:
    framework.memory.append(opp)
    framework.write_memory()
    print(f"Best opportunity: {opp.deal.product_description[:60]}... discount ${opp.discount:.2f}")
else:
    print("No new opportunity above threshold this run.")


## 6. Gradio UI (optional)

Launch the full "The Price is Right" app with log stream and opportunities table (see week8 day5 / `price_is_right.py`).

In [ ]:
from price_is_right import App

app = App()
app.run()
